In [1]:
import os
import pandas as pd
import torch
from tqdm import tqdm
from torch_geometric.data import Data
from sklearn.model_selection import train_test_split
def get_initial_data(random_state=42):
    '''
    missing_corrected.csv를 불러오고,
    데이터 스플릿까지 하는 함수
    Args:
        random_state(int, optional): 데이터 스플릿할 때 필요한 랜덤 시드
    '''
    CURDIR = os.getcwd()
    DATA_PATH = os.path.join(CURDIR, 'missing_corrected.csv')
    DATA = pd.read_csv(DATA_PATH)
    DATA.head()

    # 범주형 변수 더미화 시 train/test 간 불일치를 방지하기 위해
    # 스플릿 전에 전체 데이터 기준으로 가능한 범주를 정의하고 CategoricalDtype으로 고정
    for col in DATA.columns:
        cats = list(DATA[col].unique())
        DATA[col] = DATA[col].astype(pd.api.types.CategoricalDtype(categories=cats))

    # validation set, test set이 imputation 설계하는 데 들어가면 안 됨, 일종의 사후판단 정보가 들어갈 수 있기 때문
    y = DATA['REASONb']
    X = DATA.drop('REASONb', axis=1)
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.30, random_state=random_state
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.5, random_state=42
    )
    return X_train, X_val, X_test, y_train, y_val, y_test

In [2]:
data = get_initial_data()

In [3]:
X_train = data[0]
X_train.head()

,STFIPS,EDUC,MARSTAT,SERVICES,DETCRIM,LOS,PSOURCE,NOPRIOR,ARRESTS,EMPLOY,...,BARBFLG,SEDHPFLG,INHFLG,OTCFLG,OTHERFLG,DIVISION,REGION,IDU,ALCDRUG,CBSA2020
1139344,36,5,2,7,0,13,1,1,0,1,...,0,0,0,0,0,2,1,0,2,35620
847019,37,3,3,7,0,1,1,1,0,4,...,0,0,0,0,0,5,3,1,2,-9
1018611,34,3,1,7,7,33,7,0,0,2,...,0,0,0,0,0,2,1,0,1,45940
1340597,48,2,1,7,3,34,7,0,0,4,...,0,0,0,0,0,7,3,0,2,-9
1392195,36,2,1,7,0,37,3,0,0,3,...,0,0,0,0,0,2,1,0,3,-9


In [ ]:
def get_ad_dis_col(df:pd.DataFrame):
    '''
    admission 시의 컬럼, discharge 시의 컬럼을 나누어 리턴
    Args:
        df(pd.DataFrame): 원본 데이터프레임

    Returns: 
        (admission 시의 컬럼 list, discharge 시의 컬럼 list)
    '''
    cols = list(df.columns)

    if 'LOS' in cols:
        cols.remove('LOS')

    change = []
    change_D = []

    for i in cols:
        if i.endswith('_D'):
            change_D.append(i)
            change.append(i[:-2])
    
    ad = [i for i in cols if i not in change_D]
    dis = ad.copy()
    for i in range(len(ad)):
        if dis[i] in change:
            dis[i] = dis[i] + '_D'

    return ad, dis

In [71]:
vec = X_train.loc[1139344]
ad, dis = get_ad_dis_col(X_train)
los = vec['LOS']
admission = vec[ad].to_numpy()
discharge = vec[dis].to_numpy()
admission_t = torch.tensor(admission) 
discharge_t = torch.tensor(discharge)
admission_t.size()

torch.Size([60])

In [79]:
padding = torch.full((60,), 0)

In [80]:
padding.size()

torch.Size([60])

In [84]:
max(list(X_train['LOS'].cat.categories))

37

In [85]:
a = [1,3,4]
d = [2,3,4]
a.extend(d)

In [86]:
a

[1, 3, 4, 2, 3, 4]

In [94]:
list(range(16, X_train.shape[0], 16))

[16,
 32,
 48,
 64,
 80,
 96,
 112,
 128,
 144,
 160,
 176,
 192,
 208,
 224,
 240,
 256,
 272,
 288,
 304,
 320,
 336,
 352,
 368,
 384,
 400,
 416,
 432,
 448,
 464,
 480,
 496,
 512,
 528,
 544,
 560,
 576,
 592,
 608,
 624,
 640,
 656,
 672,
 688,
 704,
 720,
 736,
 752,
 768,
 784,
 800,
 816,
 832,
 848,
 864,
 880,
 896,
 912,
 928,
 944,
 960,
 976,
 992,
 1008,
 1024,
 1040,
 1056,
 1072,
 1088,
 1104,
 1120,
 1136,
 1152,
 1168,
 1184,
 1200,
 1216,
 1232,
 1248,
 1264,
 1280,
 1296,
 1312,
 1328,
 1344,
 1360,
 1376,
 1392,
 1408,
 1424,
 1440,
 1456,
 1472,
 1488,
 1504,
 1520,
 1536,
 1552,
 1568,
 1584,
 1600,
 1616,
 1632,
 1648,
 1664,
 1680,
 1696,
 1712,
 1728,
 1744,
 1760,
 1776,
 1792,
 1808,
 1824,
 1840,
 1856,
 1872,
 1888,
 1904,
 1920,
 1936,
 1952,
 1968,
 1984,
 2000,
 2016,
 2032,
 2048,
 2064,
 2080,
 2096,
 2112,
 2128,
 2144,
 2160,
 2176,
 2192,
 2208,
 2224,
 2240,
 2256,
 2272,
 2288,
 2304,
 2320,
 2336,
 2352,
 2368,
 2384,
 2400,
 2416,
 2432,
 244

In [95]:
for i in range(16, X_train.shape[0], 16):
    X_train.iloc[i-16:i].index

In [98]:
X_train.iloc[32-16:32].index

Index([ 771012,  539779,  347599,  561493, 1378258, 1265814,  340978,  974770,
        695699,  926463,  747864,   56359,  362426, 1207858,  602623, 1107866],
      dtype='int64')

In [99]:
import numpy as np
a = np.array([1, 2])
b = np.array([3, 4])
v = np.vstack((a, b))
print(v) # [[1 2]
        #  [3 4]]


[[1 2]
 [3 4]]


In [102]:
aas = np.array([7,8])
s = np.vstack((v,aas))
print(s)

[[1 2]
 [3 4]
 [7 8]]


In [ ]:
import numpy as np

a = np.zeros((2,), int)
b 

In [4]:
a[-1]

0

In [6]:
a = np.arange(10)
a[-1]

9